# AI Pays AI — Under a Cent

Try a priced call without paying. Read the result or payment challenge, then optionally try a funded key.

**~5 min · free · live network**

- Every call carries a price
- 402 lists the ways to pay
- One key pays from a balance

**Academy next:** [Agent Economy from Scratch](https://alexar76.github.io/aimarket-courses/agent-economy-course/)

_Tip: Runtime → Run all. Hub calls use the public AIMarket surface — no API key._


In [ ]:
# AIMarket School — AI Pays AI
# Cell 1: try a priced capability without paying. A 402 may list available payment methods.
import urllib.request, urllib.error, json, uuid

HUB = 'https://modelmarket.dev'
CALL = {
    'product_id': 'prod-platon',
    'capability_id': 'platon.random@v1',
    'source_hub': 'https://oracles.modelmarket.dev/family',
    'input': {},
}

def invoke(headers=None):
    req = urllib.request.Request(HUB + '/ai-market/v2/invoke', data=json.dumps(CALL).encode(),
                                 headers={'content-type': 'application/json', **(headers or {})})
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status, json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read().decode() or '{}')

status, body = invoke()
print('status', status, '· price $%s' % body.get('price_usd'))
if status == 402:
    print('402 Payment Required — read the payment methods offered for this call:')
    for way in body.get('payment_ways', []):
        print('  ·', way['rail'], '→ header', way['header'], way.get('start', ''))
elif 200 <= status < 300:
    print('served without payment:', json.dumps(body)[:300])
else:
    print('request refused:', json.dumps(body)[:300])


In [ ]:
# Cell 2: the free trial — the same call, marked with a visitor id you choose.
wk = json.load(urllib.request.urlopen(HUB + '/.well-known/ai-market.json', timeout=20))
trial = wk.get('free_trial') or {}
print('free trial:', trial.get('max_invokes_per_visitor'), 'priced calls, renewed', trial.get('quota_window'))
visitor = {trial.get('visitor_header', 'X-AIMarket-Sandbox-Visitor'): 'school-' + uuid.uuid4().hex[:12]}
status, body = invoke(visitor)
print('status', status, '· sandbox', body.get('sandbox'))
if status == 200:
    print('output', json.dumps(body.get('output'))[:200])
    print('signed receipt fields:', sorted((body.get('receipt') or {}).keys()))
else:
    print(body.get('error'), body.get('detail', '')[:300])


In [ ]:
# Cell 3: optional paid call using an existing funded key from /start.
# A new key can start at zero. The demo and quote checks need no key.
import getpass, os
if input("Type PAY to authorize one priced call with your Hub credit: ").strip() == "PAY":
    API_KEY = os.environ.get("AIMARKET_API_KEY") or getpass.getpass("Hub API key (hidden): ").strip()
    if API_KEY:
        status, body = invoke({"X-API-Key": API_KEY})
        print("status", status, "catalog price", body.get("price_usd"), "error", body.get("error"))
        req = urllib.request.Request(HUB + "/ai-market/v2/account", headers={"X-API-Key": API_KEY})
        try:
            with urllib.request.urlopen(req, timeout=20) as r:
                account = json.load(r)
            print("balance", account.get("balance_usd"), "spent", account.get("spent_usd"))
        except urllib.error.HTTPError as e:
            print("could not read account:", e.code)
    else:
        print("Get a key and check its balance at " + HUB + "/start")


In [ ]:
# Cell 4: seller-direct x402 quote. HEAD sends no payment header.
seller = "https://tienda.textokit.store/api/x402-ping"
try:
    with urllib.request.urlopen(urllib.request.Request(seller, method="HEAD"), timeout=20) as r:
        print("No payment challenge:", r.status)
except urllib.error.HTTPError as e:
    print("HTTP", e.code)
    if e.code == 402:
        import base64
        encoded = e.headers.get("Payment-Required")
        if encoded:
            quote = json.loads(base64.b64decode(encoded))
            for offer in quote.get("accepts", []):
                print({k: offer.get(k) for k in ("amount", "asset", "network", "payTo")})
        else:
            print("This HEAD response did not include a Payment-Required header.")


## Next

1. Open the Academy: [Agent Economy from Scratch](https://alexar76.github.io/aimarket-courses/agent-economy-course/)
2. Fill `# YOUR CODE HERE` stubs in `courselib/exercises.py`
3. `python labs/run_exercises.py --certificate "Your Name"`
4. Connect your own agent: `claude mcp add --transport http aimarket https://modelmarket.dev/mcp` — other clients and a prepaid key: https://modelmarket.dev/start
